# BurnInWatch — Structured Aging Signal Extraction v2

## Why this notebook exists

The previous extraction showed an important result:

`steadyState` records contain:
- `date`
- `timeEpoch`
- `timeDomain`

The actual engineering measurements are therefore likely nested inside `timeDomain`. The first extraction only captured `timeEpoch`, so **we must not treat the current `aging_steady_state_raw.csv` as our final feature table**.

This notebook digs into that nested structure and also fixes the SMU metadata mapping.

We will not train a model yet.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

DATA_ROOT = Path("dataset/Data").resolve()
OUT_DIR = Path("processed")
OUT_DIR.mkdir(exist_ok=True)

print(DATA_ROOT)

C:\Users\Abhishek Kuma\Desktop\BurnInWatch\dataset\Data


## 1. Correct SMU metadata mapping

The previous notebook attached `Part 11` to `device_family` and `Breakdown.csv` to `part_id`. That was reversed because of the folder depth.

The correct path interpretation is:

```text
SMU Data for new devices/
    IGBT-IRG4BC30K/
        Part 11/
            Breakdown.csv
```

So we want:

```text
device_family = IGBT-IRG4BC30K
part_id       = Part 11
test_type     = Breakdown
```

In [2]:
# ============================================================
# 1. REBUILD SMU TABLE WITH CORRECT METADATA
# ============================================================

csv_files = sorted(DATA_ROOT.rglob("*.csv"))
smu_rows = []

for path in csv_files:
    try:
        df = pd.read_csv(path, header=None)

        if df.shape[1] != 2:
            print("Unexpected:", path.relative_to(DATA_ROOT), df.shape)
            continue

        df.columns = ["measurement_1", "measurement_2"]
        df["measurement_1"] = pd.to_numeric(df["measurement_1"], errors="coerce")
        df["measurement_2"] = pd.to_numeric(df["measurement_2"], errors="coerce")

        rel = path.relative_to(DATA_ROOT)
        parts = rel.parts

        # Expected:
        # parts[0] = SMU Data for new devices
        # parts[1] = IGBT-IRG4BC30K / MOSFET-IRF520Npbf
        # parts[2] = Part XX
        # filename = Breakdown.csv / LeakageIV.csv / Turn On.csv

        device_family = parts[1] if len(parts) > 1 else "unknown"
        part_id = parts[2] if len(parts) > 2 else "unknown"
        test_type = path.stem

        tmp = df.copy()
        tmp["device_family"] = device_family
        tmp["part_id"] = part_id
        tmp["test_type"] = test_type
        tmp["source_file"] = str(rel)

        smu_rows.append(tmp)

    except Exception as e:
        print("FAILED:", path.relative_to(DATA_ROOT), repr(e))

smu_raw = pd.concat(smu_rows, ignore_index=True)

print("Rows:", len(smu_raw))
display(smu_raw.head(10))

print("\nDevice families:")
display(smu_raw["device_family"].value_counts())

print("\nParts:")
display(
    smu_raw[["device_family", "part_id"]]
    .drop_duplicates()
    .sort_values(["device_family", "part_id"])
)

smu_raw.to_csv(OUT_DIR / "smu_raw_v2.csv", index=False)

Rows: 24000


,measurement_1,measurement_2,device_family,part_id,test_type,source_file
0,545.2415,0.000010,IGBT-IRG4BC30K,Part 11,Breakdown,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...
1,550.5257,0.000004,IGBT-IRG4BC30K,Part 11,Breakdown,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...
2,551.0267,0.000004,IGBT-IRG4BC30K,Part 11,Breakdown,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...
3,551.5286,0.000004,IGBT-IRG4BC30K,Part 11,Breakdown,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...
4,552.0283,0.000004,IGBT-IRG4BC30K,Part 11,Breakdown,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...
5,552.5309,0.000004,IGBT-IRG4BC30K,Part 11,Breakdown,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...
6,553.0318,0.000004,IGBT-IRG4BC30K,Part 11,Breakdown,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...
7,553.5347,0.000004,IGBT-IRG4BC30K,Part 11,Breakdown,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...
8,554.0364,0.000004,IGBT-IRG4BC30K,Part 11,Breakdown,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...
9,554.5375,0.000004,IGBT-IRG4BC30K,Part 11,Breakdown,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...



Device families:


device_family
IGBT-IRG4BC30K       12000
MOSFET-IRF520Npbf    12000
Name: count, dtype: int64


Parts:


,device_family,part_id
0,IGBT-IRG4BC30K,Part 11
600,IGBT-IRG4BC30K,Part 12
1200,IGBT-IRG4BC30K,Part 13
1800,IGBT-IRG4BC30K,Part 14
2400,IGBT-IRG4BC30K,Part 15
3000,IGBT-IRG4BC30K,Part 16
3600,IGBT-IRG4BC30K,Part 17
4200,IGBT-IRG4BC30K,Part 18
4800,IGBT-IRG4BC30K,Part 19
5400,IGBT-IRG4BC30K,Part 1A


## 2. Inspect the nested `timeDomain`

The critical question now is:

> What actual electrical/thermal measurements are stored inside `timeDomain`?

We inspect one real `steadyState` record without assuming field names.

In [3]:
# ============================================================
# 2. INSPECT NESTED TIME DOMAIN
# ============================================================

device_mat_files = sorted(
    p for p in DATA_ROOT.rglob("*.mat")
    if "Aging Data" in str(p)
)

first_path = device_mat_files[0]
print("File:", first_path.relative_to(DATA_ROOT))

data = loadmat(first_path, squeeze_me=True, struct_as_record=False)
measurement = data["measurement"]

record = measurement.steadyState[0]
td = record.timeDomain

print("steadyState fields:", record._fieldnames)
print("\ntimeDomain type:", type(td))
print("timeDomain fields:", getattr(td, "_fieldnames", None))

if hasattr(td, "_fieldnames"):
    for field in td._fieldnames:
        value = getattr(td, field)
        print(
            f"{field:40s}",
            "type=", type(value).__name__,
            "shape=", getattr(value, "shape", None),
            "dtype=", getattr(value, "dtype", None),
            "value=", repr(value)[:250]
        )

File: Thermal Overstress Aging with Square Signal at gate and SMU data\Aging Data\Device 2\Device2  1.mat
steadyState fields: ['date', 'timeEpoch', 'timeDomain']

timeDomain type: <class 'scipy.io.matlab._mio5_params.mat_struct'>
timeDomain fields: ['supplyVoltage', 'node1Voltage', 'node2Voltage', 'collectorEmitterCurrent', 'heatSinkTemperature', 'packageTemperature', 'internalTemperature', 'ambientTemperature']
supplyVoltage                            type= float shape= None dtype= None value= nan
node1Voltage                             type= float shape= None dtype= None value= nan
node2Voltage                             type= float shape= None dtype= None value= nan
collectorEmitterCurrent                  type= float shape= None dtype= None value= nan
heatSinkTemperature                      type= float shape= None dtype= None value= nan
packageTemperature                       type= float shape= None dtype= None value= nan
internalTemperature                      type= float sha

In [4]:
# ============================================================
# 3. INSPECT NESTING RECURSIVELY FOR ONE RECORD
# ============================================================

def inspect_struct(obj, name="root", depth=0, max_depth=5):
    rows = [{
        "path": name,
        "type": type(obj).__name__,
        "shape": str(getattr(obj, "shape", None)),
        "dtype": str(getattr(obj, "dtype", None)),
        "value_preview": repr(obj)[:180]
    }]

    if depth >= max_depth:
        return rows

    if hasattr(obj, "_fieldnames"):
        for field in obj._fieldnames:
            try:
                child = getattr(obj, field)
                rows.extend(
                    inspect_struct(
                        child,
                        f"{name}.{field}",
                        depth + 1,
                        max_depth
                    )
                )
            except Exception as e:
                rows.append({
                    "path": f"{name}.{field}",
                    "type": "ERROR",
                    "shape": "",
                    "dtype": "",
                    "value_preview": str(e)
                })

    return rows

nested = pd.DataFrame(
    inspect_struct(record, "steadyState[0]", max_depth=6)
)

display(nested)
nested.to_csv(OUT_DIR / "first_steady_state_nested_structure.csv", index=False)

,path,type,shape,dtype,value_preview
0,steadyState[0],mat_struct,None,None,<scipy.io.matlab._mio5_params.mat_struct objec...
1,steadyState[0].date,str,None,None,'12/31/1903 4:00:00 PM'
2,steadyState[0].timeEpoch,int,None,None,0
3,steadyState[0].timeDomain,mat_struct,None,None,<scipy.io.matlab._mio5_params.mat_struct objec...
4,steadyState[0].timeDomain.supplyVoltage,float,None,None,nan
5,steadyState[0].timeDomain.node1Voltage,float,None,None,nan
6,steadyState[0].timeDomain.node2Voltage,float,None,None,nan
7,steadyState[0].timeDomain.collectorEmitterCurrent,float,None,None,nan
8,steadyState[0].timeDomain.heatSinkTemperature,float,None,None,nan
9,steadyState[0].timeDomain.packageTemperature,float,None,None,nan


## 4. Discover all fields inside `timeDomain`

We now scan all devices and collect the nested field names and Python types.

This is still discovery; no data is modified.

In [5]:
# ============================================================
# 4. FIELD INVENTORY FOR TIME DOMAIN
# ============================================================

td_field_types = {}

for path in device_mat_files:
    try:
        data = loadmat(path, squeeze_me=True, struct_as_record=False)
        measurement = data["measurement"]
        steady = getattr(measurement, "steadyState", [])

        # Inspect up to the first 50 records per file first.
        # If the structure is consistent, this is enough to discover fields.
        for record in list(steady)[:50]:
            td = getattr(record, "timeDomain", None)

            if not hasattr(td, "_fieldnames"):
                continue

            for field in td._fieldnames:
                value = getattr(td, field)
                td_field_types.setdefault(field, set()).add(type(value).__name__)

    except Exception as e:
        print("FAILED:", path.relative_to(DATA_ROOT), repr(e))

td_fields = pd.DataFrame([
    {
        "timeDomain_field": field,
        "python_types": ", ".join(sorted(types))
    }
    for field, types in sorted(td_field_types.items())
])

display(td_fields)
td_fields.to_csv(OUT_DIR / "timeDomain_field_inventory.csv", index=False)

,timeDomain_field,python_types
0,ambientTemperature,float
1,collectorEmitterCurrent,float
2,heatSinkTemperature,float
3,internalTemperature,float
4,node1Voltage,float
5,node2Voltage,float
6,packageTemperature,float
7,supplyVoltage,float


In [6]:
# ============================================================
# 5. SHOW ACTUAL VALUES FOR DISCOVERED TIME-DOMAIN FIELDS
# ============================================================

if not td_fields.empty:
    data = loadmat(
        first_path,
        squeeze_me=True,
        struct_as_record=False
    )

    record = data["measurement"].steadyState[0]
    td = record.timeDomain

    preview_rows = []

    for field in td._fieldnames:
        value = getattr(td, field)

        try:
            arr = np.asarray(value)

            preview_rows.append({
                "field": field,
                "type": type(value).__name__,
                "shape": str(getattr(value, "shape", None)),
                "dtype": str(getattr(value, "dtype", None)),
                "value": repr(value)[:300],
                "numeric_min": (
                    float(np.nanmin(arr.astype(float)))
                    if np.issubdtype(arr.dtype, np.number) else np.nan
                ),
                "numeric_max": (
                    float(np.nanmax(arr.astype(float)))
                    if np.issubdtype(arr.dtype, np.number) else np.nan
                )
            })
        except Exception as e:
            preview_rows.append({
                "field": field,
                "type": type(value).__name__,
                "shape": str(getattr(value, "shape", None)),
                "dtype": str(getattr(value, "dtype", None)),
                "value": repr(value)[:300],
                "numeric_min": np.nan,
                "numeric_max": np.nan
            })

    td_preview = pd.DataFrame(preview_rows)
    display(td_preview)

    td_preview.to_csv(
        OUT_DIR / "timeDomain_value_preview.csv",
        index=False
    )

,field,type,shape,dtype,value,numeric_min,numeric_max
0,supplyVoltage,float,None,None,nan,NaN,NaN
1,node1Voltage,float,None,None,nan,NaN,NaN
2,node2Voltage,float,None,None,nan,NaN,NaN
3,collectorEmitterCurrent,float,None,None,nan,NaN,NaN
4,heatSinkTemperature,float,None,None,nan,NaN,NaN
5,packageTemperature,float,None,None,nan,NaN,NaN
6,internalTemperature,float,None,None,nan,NaN,NaN
7,ambientTemperature,float,None,None,nan,NaN,NaN


## 6. Inspect transient records too

The transient records may contain different measurements from steady-state records. We should know what they contain before deciding what to use.

In [7]:
# ============================================================
# 6. TRANSIENT FIELD INVENTORY
# ============================================================

transient_field_types = {}

for path in device_mat_files:
    try:
        data = loadmat(path, squeeze_me=True, struct_as_record=False)
        measurement = data["measurement"]
        transient = getattr(measurement, "transient", [])

        for record in list(transient)[:50]:
            if not hasattr(record, "_fieldnames"):
                continue

            for field in record._fieldnames:
                value = getattr(record, field)
                transient_field_types.setdefault(
                    field, set()
                ).add(type(value).__name__)

    except Exception as e:
        print("FAILED:", path.relative_to(DATA_ROOT), repr(e))

transient_fields = pd.DataFrame([
    {
        "transient_field": field,
        "python_types": ", ".join(sorted(types))
    }
    for field, types in sorted(transient_field_types.items())
])

display(transient_fields)
transient_fields.to_csv(
    OUT_DIR / "transient_field_inventory.csv",
    index=False
)

,transient_field,python_types
0,date,str
1,frequencyDomain,mat_struct
2,timeDomain,mat_struct
3,timeEpoch,float


## 7. Time behavior of the large DC experiment

The raw plots show a strongly switched signal: collector current alternates between low and high states, while package temperature stays around a relatively stable high-temperature band for most of the run.

That means **raw point-by-point values are not yet our final ML samples**.

We will later use windows/cycles/statistics rather than treating every electrical switching point as an independent component observation.

In [8]:
# ============================================================
# 7. CREATE RELATIVE TIME FOR DC EXPERIMENT
# ============================================================

dc_path = sorted(
    p for p in DATA_ROOT.rglob("*.mat")
    if "Thermal Overstress Aging with DC at gate" in str(p)
)[0]

dc = loadmat(dc_path, squeeze_me=True, struct_as_record=False)

dc_raw = pd.DataFrame({
    "time_raw": np.asarray(dc["TIME"]).squeeze(),
    "gate_voltage": np.asarray(dc["GATE_VOLTAGE"]).squeeze(),
    "collector_voltage": np.asarray(dc["COLLECTOR_VOLTAGE"]).squeeze(),
    "heat_sink_temp": np.asarray(dc["HEAT_SINK_TEMP"]).squeeze(),
    "gate_current": np.asarray(dc["GATE_CURRENT"]).squeeze(),
    "collector_current": np.asarray(dc["COLLECTOR_CURRENT"]).squeeze(),
    "package_temp": np.asarray(dc["PACKAGE_TEMP"]).squeeze(),
})

dc_raw["relative_time"] = (
    dc_raw["time_raw"] - dc_raw["time_raw"].iloc[0]
)

dt = dc_raw["relative_time"].diff()

print("Relative duration:", dc_raw["relative_time"].iloc[-1])
print("Median sample interval:", dt.median())
print("Mean sample interval:", dt.mean())
print("Minimum interval:", dt.min())
print("Maximum interval:", dt.max())
print("Non-positive intervals:", int((dt.dropna() <= 0).sum()))

display(dt.describe())

Relative duration: 15083.950000286102
Median sample interval: 0.05000019073486328
Mean sample interval: 0.05000000000094837
Minimum interval: 0.04999971389770508
Maximum interval: 0.05000019073486328
Non-positive intervals: 0


count    3.016790e+05
mean     5.000000e-02
std      2.336017e-07
min      4.999971e-02
25%      4.999971e-02
50%      5.000019e-02
75%      5.000019e-02
max      5.000019e-02
Name: relative_time, dtype: float64

In [9]:
# ============================================================
# 8. SAVE DC RAW DATA V2
# ============================================================

dc_raw["sample_index"] = np.arange(len(dc_raw))
dc_raw["source_file"] = str(dc_path.relative_to(DATA_ROOT))

dc_raw.to_csv(
    OUT_DIR / "dc_aging_raw_v2.csv",
    index=False
)

print("Saved:", OUT_DIR / "dc_aging_raw_v2.csv")

Saved: processed\dc_aging_raw_v2.csv


# STOP POINT

Run this notebook and send the following outputs:

### Most important
1. `timeDomain fields`
2. `first_steady_state_nested_structure.csv` / table
3. `timeDomain_value_preview`
4. `transient_field_inventory`

### Also send
5. corrected SMU metadata table
6. DC relative-time statistics

**Do not train anything yet.**

Once we see the actual `timeDomain` fields, we can write the first real extraction code for the engineering signals and decide what the canonical BurnInWatch dataset should contain.